In [2]:
!pip install bert_score transformers

In [3]:
from google.colab import drive
import os

In [4]:
# Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Paths
source_sample_dir = 'YOUR_PATH'
output_dir = 'YOUR_PATH'

In [6]:
all_sources = []
all_summaries = []
filenames = []

# Get sorted list of files to ensure matching
file_list = sorted([f for f in os.listdir(output_dir) if f.endswith('.txt')])

for file_name in file_list:
    # Remove "_summary" from BenSum filenames
    src_file = file_name.replace("_summary", "")

    src_path = os.path.join(source_sample_dir, src_file)
    sum_path = os.path.join(output_dir, file_name)

    if os.path.exists(src_path):
        with open(src_path, 'r', encoding='utf-8') as f:
            all_sources.append(f.read())
        with open(sum_path, 'r', encoding='utf-8') as f:
            all_summaries.append(f.read())
        filenames.append(file_name)

print(f" Loaded {len(all_summaries)} file pairs successfully.")
# Quick check to ensure they match
print(f"First file check: {filenames[0]}")

 Loaded 242 file pairs successfully.
First file check: acts_1981-01-29_Presidential_Elections_Special_Provisions_si_summary.txt


In [ ]:
from bert_score import score
import pandas as pd

print("Starting BERTScore calculation...")

# Compare Summary to Source (Self-BERTScore)
# 'lang="any"' with 'bert-base-multilingual-cased' - best setup for Sinhala
P, R, F1 = score(all_summaries, all_sources,
                 lang="any",
                 model_type="bert-base-multilingual-cased",
                 device="cuda")

# Create a results table
bert_results = pd.DataFrame({
    "File": filenames,
    "Precision": P.tolist(),
    "Recall": R.tolist(),
    "F1_Score": F1.tolist()
})

# Save results to CSV
output_path = 'YOUR_PATH'
bert_results.to_csv(output_path, index=False)

print("\n" + "="*30)
print(f" FINAL AVERAGE BERTSCORE (F1): {F1.mean().item():.4f}")
print(f"Results saved to: {output_path}")
print("="*30)

Starting BERTScore calculation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 FINAL AVERAGE BERTSCORE (F1): 0.8031
Results saved to: /content/drive/MyDrive/SinBrief_Implementation/Evaluations/Llama/Result_Sheets/bertscore_results.csv
